# 1. Making zoom initial conditions (with MUSIC-2)

Before we can run a zoom simulation, we need initial conditions to run... this normal entails choosing a parent simulation (a large low resolution simulation), identifying your region/regions of interest, and generating particle distributions at high resolution in the region of interest and low resolution everywhere else (with resolution degrading the further from the region of interest you go). 

In this notebook we will use MUSIC-2 to generate some zoom initial conditions for a TESSERA simulation (other initial condition pipelines are available). Shell commands are shown as Markdown blocks to copy into a terminal; only the Python inspection cells run in the notebook.

In [ ]:
from pathlib import Path
import h5py
import matplotlib.pyplot as plt
import numpy as np


# Some helpful constants for paths...
ROOT = Path('..')
CONFIG = ROOT / 'configs' / 'music.conf'
IC = ROOT / 'data' / 'zoom_ics.hdf5'
print(f'Working in {ROOT.resolve()}')

## Our target

We will use region `P00000/R0000` from the TESSERA simulation suite. It is an underdense region selected from parent volume `P00000`, with density contrast $\delta=-0.0815$ and an effective radius of about $57.5$ Mpc.

The 35,608 parent particles belonging to that region were traced back to the initial conditions. Their positions are stored in `data/tessera_P00000_R0000_region_points.txt`; MUSIC-2 fits the ellipsoid that tells it where to add the extra resolution. The original selection metadata is included beside it.

## The MUSIC-2 setup

The full configuration is printed below. The most important values are the three grid levels:

- `levelmin = 6` gives the coarse $64^3$ particle load covering the whole box.
- `levelmin_TF = 8` evaluates the transfer fields on a reduced $256^3$ grid for the workshop.
- `levelmax = 10` gives effective $1024^3$ spacing inside the ellipsoid.

We use second-order Lagrangian perturbation theory (`use_2LPT = yes`) to reduce transients from starting the simulation at a finite redshift.

In [ ]:
print(CONFIG.read_text())

Each extra finest level divides the particle mass by eight. The production TESSERA zoom reaches level 13; we stop at level 10 for this workshop:

| Finest level | Finest DMO particle mass [$h^{-1} M_\odot$] | Finest DMO particle mass [$M_\odot$] |
|---:|---:|---:|
| 10 | $3.99\times10^{11}$ | $5.96\times10^{11}$ |
| 11 | $4.99\times10^{10}$ | $7.45\times10^{10}$ |
| 12 | $6.24\times10^9$ | $9.31\times10^9$ |
| 13 | $7.80\times10^8$ | $1.16\times10^9$ |

We will use level 10. This is deliberately much coarser than the production run, but still gives us a nested particle load for exploring SWIFT's zoom geometry.

## Build MUSIC-2

MUSIC-2 uses OpenMP, so this makes one shared-memory executable rather than an MPI build. Copy the appropriate block below into a terminal from the repository root.

**Local machine**

```bash
scripts/build_music2.sh
```

**COSMA8 login node**

```bash
module load intel_comp/2024.2.0 compiler-rt tbb compiler mpi ucx/1.17.0 parallel_hdf5/1.14.4 fftw/3.3.10 parmetis/4.0.3-64bit gsl/2.8
scripts/build_music2.sh
```

## Generate the ICs

MUSIC-2 uses the thread count set by `NumThreads` in the `[execution]` section of `configs/music.conf`. Change that value to suit your machine, then copy the appropriate command into a terminal from the repository root.

**Local machine**

```bash
scripts/run_music.sh
```

**COSMA8**

```bash
sbatch submission_scripts/music_cosma8.slurm
```

If MUSIC-2 ran on an HPC but this notebook is running locally, copy the IC file back from a local terminal in the repository root:

```bash
rsync -avz -P <fill in your username>@<fill in the HPC domain>:<fill in the absolute path to the repository>/data/zoom_ics.hdf5 data/
```

## What did MUSIC-2 make?

Before handing the file to SWIFT, let's check that it contains high-resolution dark matter (`PartType1`) and coarse dark matter (`PartType2`), and that the positions, masses, and IDs look sensible.

In [ ]:
with h5py.File(IC, 'r') as handle:
    box = float(np.atleast_1d(handle['Header'].attrs['BoxSize'])[0])
    print(f'Box size: {box:g} Mpc')
    for part_type in ('PartType1', 'PartType2'):
        group = handle[part_type]
        coordinates = group['Coordinates'][:]
        masses = group['Masses'][:]
        ids = group['ParticleIDs'][:]
        print(f'{part_type}: {len(ids):,} particles, mass range {masses.min():.3g}--{masses.max():.3g}')

In [ ]:
with h5py.File(IC, 'r') as handle:
    high = handle['PartType1']['Coordinates'][:]
    coarse = handle['PartType2']['Coordinates'][:]

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for xyz, colour, label in ((coarse, '0.65', 'coarse DM'), (high, 'tab:blue', 'high-resolution DM')):
    stride = max(1, len(xyz) // 100_000)
    axes[0].scatter(xyz[::stride, 0], xyz[::stride, 1], s=0.2, c=colour, label=label)
    axes[1].scatter(xyz[::stride, 0], xyz[::stride, 1], s=0.4, c=colour, label=label)

padding = 0.15 * np.ptp(high[:, :2], axis=0)
axes[1].set_xlim(high[:, 0].min() - padding[0], high[:, 0].max() + padding[0])
axes[1].set_ylim(high[:, 1].min() - padding[1], high[:, 1].max() + padding[1])
axes[0].set_title('The full box')
axes[1].set_title('The high-resolution region')
for ax in axes:
    ax.set(xlabel='x [Mpc]', ylabel='y [Mpc]')
    ax.set_aspect('equal')
axes[0].legend(markerscale=15)
fig.tight_layout()
plt.show()

The left panel should show the coarse particles filling the whole box. The right panel should reveal the much denser ellipsoidal Lagrangian region. Have a look at the MUSIC log in `runs/music/music.log` if your particle counts differ substantially from the estimate above.

We now have everything needed to explore SWIFT's zoom cell geometry.